## Goal

Take the RAG chain from Tutorial 1 and add:

1. evaluation hooks
2. error logging
3. API exposure for reproducible testing

## Step 1: Define an Evaluation Dataset


In [ ]:
import pandas as pd

eval_set = pd.DataFrame([
	{"question": "What is modular RAG?", "reference": "A componentized retrieval-generation system."},
	{"question": "What does efSearch control?", "reference": "Query-time search breadth in HNSW."},
])

print(eval_set)


## Step 2: Build a Scoring Function


In [ ]:
def overlap_relevance(answer: str, reference: str) -> float:
	a = set(answer.lower().split())
	r = set(reference.lower().split())
	return len(a.intersection(r)) / max(1, len(r))


## Step 3: Evaluate the RAG Chain


In [ ]:
def evaluate_rag(ask_fn, dataset: pd.DataFrame) -> pd.DataFrame:
	rows = []
	for _, row in dataset.iterrows():
		q = row["question"]
		ref = row["reference"]
		ans = ask_fn(q)
		rows.append({
			"question": q,
			"reference": ref,
			"answer": ans,
			"relevance": overlap_relevance(ans, ref),
		})
	return pd.DataFrame(rows)

# results = evaluate_rag(ask_rag, eval_set)
# print(results)


## Step 4: Log Failure Cases


In [ ]:
def tag_failure(answer: str) -> str:
	lower = answer.lower()
	if "insufficient" in lower:
		return "abstained"
	if len(answer.strip()) < 20:
		return "under_answered"
	return "needs_review"


## Step 5: Expose Endpoint with FastAPI + LangServe


In [ ]:
from fastapi import FastAPI
from langserve import add_routes

app = FastAPI(title="M05-RAG-Service")

# If chain is your LCEL chain object:
# add_routes(app, chain, path="/rag")

print("Endpoint scaffold ready at /rag")


## Step 6: Run Quick Smoke Tests

Use a few prompt classes:

1. factual direct questions
2. ambiguous questions
3. out-of-scope questions

Document when the model should abstain.

## Output Artifact

Create a short appendix with:

1. evaluation table (`question`, `answer`, `relevance`)
2. failure tags and examples
3. endpoint route definition

These become direct inputs for the Methods and Results sections in Assignment 4.
